# Phase 3: Feature Interpretation & Cross-Algorithm Comparison

Analyze SAE features trained in Phase 2:
1. **Per-algorithm**: reconstruction quality, concept correlations, monosemanticity
2. **Cross-algorithm**: which features/concepts are universal vs algorithm-specific
3. **Linear probes**: baseline for concept decodability
4. **Summary**: consolidated tables and heatmaps

In [ ]:
import sys
from pathlib import Path

if Path("../data").exists() and ".." not in sys.path:
    sys.path.insert(0, "..")
elif Path("data").exists() and "." not in sys.path:
    sys.path.insert(0, ".")

try:
    import torch_geometric.data.data as _pyg_data
    if not hasattr(_pyg_data, 'DataEdgeAttr'):
        _pyg_data.DataEdgeAttr = type('DataEdgeAttr', (), {})
        _pyg_data.DataTensorAttr = type('DataTensorAttr', (), {})
except Exception:
    pass

import os
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

from data import AVAILABLE_ALGORITHMS
from interp import (
    BatchTopKSAE, FeatureAnalyzer,
    fit_linear_probes, save_probe_results, load_probe_results,
)
from utils.evaluation import evaluate_sae, compute_concept_analysis
from utils.correlation import pearson_correlation_matrix, find_monosemantic_features

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## Configuration

In [ ]:
ALGORITHMS = AVAILABLE_ALGORITHMS
LOCAL_DEBUG = False

if LOCAL_DEBUG:
    HIDDEN_DIM = 32
    NUM_LAYERS = 2
else:
    HIDDEN_DIM = 128
    NUM_LAYERS = 4

EXPANSION_FACTOR = 4
SAE_K = 16 if LOCAL_DEBUG else 32
SAE_BATCH_SIZE = 256
MONO_THRESH = 0.4
N_EVAL = 10_000

IN_COLAB = "COLAB_GPU" in os.environ
REPO_ROOT = Path("..") if Path("../data").exists() else Path(".")
if IN_COLAB:
    RESULTS_ROOT = Path("/content/drive/MyDrive/nar-mechinterp/results/multi_algo")
else:
    RESULTS_ROOT = REPO_ROOT / "results" / "multi_algo"

dict_size = HIDDEN_DIM * EXPANSION_FACTOR
print(f"Results root: {RESULTS_ROOT}")
print(f"Algorithms: {ALGORITHMS}")

---
## 1. Per-Algorithm SAE Analysis

For each algorithm: load activations + SAE + concept labels, compute
reconstruction quality and concept correlations.

In [ ]:
algo_results = {}

for algo in ALGORITHMS:
    algo_dir = RESULTS_ROOT / algo
    act_path = algo_dir / "activations.pt"
    sae_path = algo_dir / "sae.pt"
    lbl_path = algo_dir / "concept_labels.pt"
    corr_path = algo_dir / "correlation_results.pt"

    if not sae_path.exists():
        print(f"[{algo}] SAE not found, skipping.")
        continue

    # Load from cached results if available
    if corr_path.exists():
        cached = torch.load(corr_path, weights_only=False)
        algo_results[algo] = cached
        print(f"[{algo}] Loaded cached results: "
              f"{len(cached['mono'])} mono, {len(cached['dead'])} dead")
        continue

    print(f"\n[{algo}] Analyzing...")

    # Load SAE
    sae = BatchTopKSAE(input_dim=HIDDEN_DIM, dict_size=dict_size, k=SAE_K).to(device)
    ckpt = torch.load(sae_path, map_location=device, weights_only=True)
    sae.load_state_dict(ckpt["state_dict"])
    sae.eval()

    # Load activations
    activations = torch.load(act_path)

    # Evaluate reconstruction
    eval_result = evaluate_sae(sae, activations, batch_size=SAE_BATCH_SIZE,
                               device=device, max_samples=N_EVAL)
    print(f"  FVE={eval_result['fve']:.4f} | Dead={eval_result['dead_count']}/{dict_size} "
          f"({eval_result['dead_pct']:.1f}%) | L0={eval_result['l0_mean']:.1f}")

    # Concept correlations
    concept_data = torch.load(lbl_path, map_location="cpu", weights_only=True)
    concept_labels = {k: v for k, v in concept_data["labels"].items()}
    concept_names = list(concept_labels.keys())

    concept_result = compute_concept_analysis(
        eval_result["features"], concept_labels, concept_names,
        mono_thresh=MONO_THRESH, max_samples=N_EVAL,
    )

    result = {
        "fve": eval_result["fve"],
        "mse": eval_result["mse"],
        "dead_count": eval_result["dead_count"],
        "dead_pct": eval_result["dead_pct"],
        "l0_mean": eval_result["l0_mean"],
        "cm": concept_result["cm"],
        "concept_names": concept_result["concept_names"],
        "mono": concept_result["mono"],
        "dead": concept_result["dead"],
    }
    algo_results[algo] = result

    # Save
    torch.save(result, corr_path)
    print(f"  Mono: {len(result['mono'])} | Concepts: {concept_names}")

    del sae, activations, eval_result, concept_result
    torch.cuda.empty_cache()

print(f"\nAnalyzed {len(algo_results)} algorithms.")

## 2. Summary Table

In [ ]:
print(f"{'Algorithm':25s} | {'FVE':>6s} | {'Dead%':>6s} | {'L0':>5s} | {'Mono':>5s} | {'Top concepts'}")
print("-" * 100)
for algo in ALGORITHMS:
    if algo not in algo_results:
        continue
    r = algo_results[algo]
    cm = r["cm"]
    names = r["concept_names"]
    # Top concept by max |r|
    max_corrs = [(names[j], cm[:, j].abs().max().item()) for j in range(len(names))]
    max_corrs.sort(key=lambda x: -x[1])
    top_str = ", ".join(f"{n}={v:.2f}" for n, v in max_corrs[:3])
    print(f"{algo:25s} | {r['fve']:6.4f} | {r['dead_pct']:5.1f}% | "
          f"{r['l0_mean']:5.1f} | {len(r['mono']):5d} | {top_str}")

## 3. Per-Algorithm Concept Correlation Heatmaps

In [ ]:
analyzed = [a for a in ALGORITHMS if a in algo_results]
n = len(analyzed)
n_cols = min(3, n)
n_rows = (n + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 5 * n_rows))
axes = np.array(axes).reshape(-1)

for idx, algo in enumerate(analyzed):
    r = algo_results[algo]
    cm = r["cm"].numpy()
    names = r["concept_names"]

    # Show top 20 features by max correlation
    max_per_feat = np.abs(cm).max(axis=1)
    top_k = min(20, cm.shape[0])
    top_idxs = np.argsort(max_per_feat)[-top_k:][::-1]

    ax = axes[idx]
    im = ax.imshow(cm[top_idxs], cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
    ax.set_yticks(range(len(top_idxs)))
    ax.set_yticklabels([f"F{i}" for i in top_idxs], fontsize=7)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=45, ha='right', fontsize=8)
    ax.set_title(algo, fontsize=11)
    plt.colorbar(im, ax=ax, shrink=0.8)

for idx in range(n, len(axes)):
    axes[idx].set_visible(False)

fig.suptitle("Concept Correlations (top features per algorithm)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 4. Cross-Algorithm Comparison

Compare which algorithmic concepts are universally well-captured by SAE features
vs which are algorithm-specific.

In [ ]:
# Best max |r| per concept per algorithm
print(f"{'Concept':25s}", end="")
for algo in analyzed:
    print(f" | {algo[:10]:>10s}", end="")
print()
print("-" * (25 + 13 * len(analyzed)))

# Collect all concepts across algorithms
all_concepts = set()
for algo in analyzed:
    all_concepts.update(algo_results[algo]["concept_names"])

for concept in sorted(all_concepts):
    print(f"{concept:25s}", end="")
    for algo in analyzed:
        r = algo_results[algo]
        if concept in r["concept_names"]:
            j = r["concept_names"].index(concept)
            max_r = r["cm"][:, j].abs().max().item()
            print(f" | {max_r:10.3f}", end="")
        else:
            print(f" | {'---':>10s}", end="")
    print()

In [ ]:
# Build a matrix: rows=concepts (across all algos), cols=algorithms
all_concepts_list = sorted(all_concepts)
cross_matrix = np.full((len(all_concepts_list), len(analyzed)), np.nan)

for j, algo in enumerate(analyzed):
    r = algo_results[algo]
    for i, concept in enumerate(all_concepts_list):
        if concept in r["concept_names"]:
            cidx = r["concept_names"].index(concept)
            cross_matrix[i, j] = r["cm"][:, cidx].abs().max().item()

fig, ax = plt.subplots(figsize=(max(8, len(analyzed) * 1.2), max(6, len(all_concepts_list) * 0.4)))
im = ax.imshow(cross_matrix, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)
ax.set_xticks(range(len(analyzed)))
ax.set_xticklabels(analyzed, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(len(all_concepts_list)))
ax.set_yticklabels(all_concepts_list, fontsize=8)
ax.set_title("Max |Pearson r| per concept per algorithm")
plt.colorbar(im, ax=ax, shrink=0.7)

# Annotate cells
for i in range(len(all_concepts_list)):
    for j in range(len(analyzed)):
        val = cross_matrix[i, j]
        if not np.isnan(val):
            ax.text(j, i, f"{val:.2f}", ha='center', va='center', fontsize=7,
                    color='white' if val > 0.5 else 'black')

plt.tight_layout()
plt.show()

---
## 5. Linear Probes Per Algorithm

Fit logistic regression probes per layer and concept as a ceiling comparison.

In [ ]:
probe_path = RESULTS_ROOT / "linear_probes.pt"

if probe_path.exists():
    all_probes = torch.load(probe_path, weights_only=False)
    print(f"Loaded probe results for {len(all_probes)} algorithms")
else:
    all_probes = {}

    for algo in ALGORITHMS:
        algo_dir = RESULTS_ROOT / algo
        pla_path = algo_dir / "per_layer_activations.pt"
        lbl_path = algo_dir / "concept_labels.pt"

        if not pla_path.exists() or not lbl_path.exists():
            print(f"[{algo}] Missing per-layer activations or labels, skipping probes.")
            continue

        print(f"\n[{algo}] Fitting linear probes...")
        per_layer_acts = torch.load(pla_path)
        concept_data = torch.load(lbl_path, map_location="cpu", weights_only=True)

        probe_output = fit_linear_probes(
            per_layer_acts=per_layer_acts,
            concept_labels=concept_data["labels"],
            num_layers=NUM_LAYERS,
            n_samples=20_000,
            verbose=True,
        )
        all_probes[algo] = probe_output
        del per_layer_acts

    torch.save(all_probes, probe_path)
    print(f"\nSaved probe results -> {probe_path}")

## 6. Probe AUROC Heatmap

In [ ]:
probed_algos = [a for a in ALGORITHMS if a in all_probes]

if not probed_algos:
    print("No probe results available.")
else:
    for algo in probed_algos:
        pr = all_probes[algo]
        concepts = pr["concepts"]

        auroc_matrix = np.full((len(concepts), NUM_LAYERS), np.nan)
        for i, concept in enumerate(concepts):
            for li in range(NUM_LAYERS):
                key = (li, concept)
                if key in pr["results"]:
                    auroc_matrix[i, li] = pr["results"][key]["auroc"]

        fig, ax = plt.subplots(figsize=(max(4, NUM_LAYERS * 1.5), max(3, len(concepts) * 0.5)))
        im = ax.imshow(auroc_matrix, cmap='YlGnBu', vmin=0.5, vmax=1.0, aspect='auto')
        ax.set_xticks(range(NUM_LAYERS))
        ax.set_xticklabels([f"L{i}" for i in range(NUM_LAYERS)])
        ax.set_yticks(range(len(concepts)))
        ax.set_yticklabels(concepts, fontsize=9)
        ax.set_title(f"Linear Probe AUROC — {algo}")
        plt.colorbar(im, ax=ax, shrink=0.8)

        for i in range(len(concepts)):
            for j in range(NUM_LAYERS):
                val = auroc_matrix[i, j]
                if not np.isnan(val):
                    ax.text(j, i, f"{val:.2f}", ha='center', va='center', fontsize=8)

        plt.tight_layout()
        plt.show()

---
## 7. Monosemantic Feature Detail

For each algorithm, show the monosemantic features and their concept associations.

In [ ]:
for algo in analyzed:
    r = algo_results[algo]
    mono = r["mono"]
    cm = r["cm"]
    names = r["concept_names"]

    if not mono:
        print(f"\n[{algo}] No monosemantic features found.")
        continue

    print(f"\n{'='*60}")
    print(f"{algo}: {len(mono)} monosemantic features")
    print(f"{'='*60}")
    for feat_idx in mono[:15]:
        corrs = {names[j]: cm[feat_idx, j].item() for j in range(len(names))}
        best = max(corrs, key=lambda k: abs(corrs[k]))
        print(f"  Feature {feat_idx:4d} -> {best:25s} (r={corrs[best]:+.3f})")

---
## 8. Consolidated Summary

Combined table with SAE metrics and probe AUROC across all algorithms.

In [ ]:
print("=" * 100)
print("CONSOLIDATED SUMMARY")
print("=" * 100)

print(f"\n{'Algorithm':25s} | {'FVE':>6s} | {'Dead%':>6s} | {'Mono':>5s} | {'Best concept':>25s} | {'Max|r|':>6s} | {'Probe AUROC':>11s}")
print("-" * 100)

for algo in ALGORITHMS:
    if algo not in algo_results:
        continue
    r = algo_results[algo]
    cm = r["cm"]
    names = r["concept_names"]

    # Best concept
    best_j = cm.abs().max(dim=0).values.argmax().item()
    best_r = cm[:, best_j].abs().max().item()
    best_concept = names[best_j]

    # Best probe AUROC
    if algo in all_probes:
        pr = all_probes[algo]
        aurocs = [v["auroc"] for v in pr["results"].values()]
        best_auroc = max(aurocs) if aurocs else float('nan')
        auroc_str = f"{best_auroc:.3f}"
    else:
        auroc_str = "N/A"

    print(f"{algo:25s} | {r['fve']:6.4f} | {r['dead_pct']:5.1f}% | "
          f"{len(r['mono']):5d} | {best_concept:>25s} | {best_r:6.3f} | {auroc_str:>11s}")

print("\nDone.")